# SilenceBreaker — Walkthrough Notebook

Multi-Agent, Risk-Aware, RAG-Based Support Assistant for Abuse-Related Help-Seeking.

**Ethics & scope.** This is a prototype information/triage tool. It does **not** give legal, medical, or professional advice and does **not** diagnose. High-risk inputs are routed to *seek immediate local help*.

This notebook walks through each component end to end with explanations, so it can be submitted as the reproducible notebook deliverable.


## 0. Setup
Install dependencies (run once). On Colab, restart the runtime afterwards.


In [ ]:
# !pip install -q -r ../requirements.txt
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `import src...` works from notebooks/


## 1. Datasets
We use two assignment-approved Hugging Face datasets:
- `dair-ai/emotion` — emotion classification → distress level (Agent 1 & 4).
- `google/jigsaw_toxicity_pred` — toxic/harassing text → binary abuse detector (Agent 1).


In [ ]:
from datasets import load_dataset
emo = load_dataset('dair-ai/emotion')
print(emo)
print(emo['train'][0])
print('labels:', emo['train'].features['label'].names)


## 2. Pre-processing
Light, meaning-preserving cleaning (keep punctuation, negations, emotional words).


In [ ]:
from src.preprocess import clean_text, EMOTION_TO_DISTRESS
print(clean_text('  My BOSS  threatened me!!! http://x.com  '))
print(EMOTION_TO_DISTRESS)


## 3. Agent 1 — Situation Listener
Returns abuse **category** (few-shot LLM / rule fallback), **distress** (emotion model), and **is_abuse** (trained DistilBERT, optional).


In [ ]:
from src.agents.listener import analyze
analyze('My manager keeps sending inappropriate messages and threatens my job if I report it.')


## 4. Build the RAG index
Chunk the curated knowledge base in `data/kb/`, embed with MiniLM, and store in FAISS.


In [ ]:
from src.rag.build_index import build
build()


## 5. Agent 2 — Retriever
Top-k cosine-similarity retrieval over the knowledge base.


In [ ]:
from src.agents.retriever import Retriever
r = Retriever()
for d in r.retrieve('how do I report harassment at work', k=3):
    print(round(d['score'],3), d['source'])


## 6. Agent 4 — Risk Evaluator
Combines distress with danger-cue rules → low / medium / high.


In [ ]:
from src.agents.risk import evaluate
print(evaluate('he has a weapon and is here right now', 'high', True))
print(evaluate('I am a bit stressed about work', 'low', False))


## 7. Agent 3 — Safety Planner
LLM grounded ONLY on retrieved evidence. In OFFLINE mode a template is used.


In [ ]:
from src.agents.planner import plan
ev = r.retrieve('report harassment at work', k=3)
print(plan('My manager threatens my job if I report his messages.',
           'workplace_harassment', 'medium', 'medium', ev))


## 8. Full multi-agent pipeline (LangGraph)
listener → retriever → risk → planner.


In [ ]:
from src.graph import run
import json
out = run('He shoved me again last night and I am scared to go home tonight.')
print(json.dumps({k:v for k,v in out.items() if k!='evidence'}, indent=2))


## 9. Evaluation
Retrieval quality, category/risk accuracy, faithfulness, and the ablation study.


In [ ]:
from evaluation.eval_retrieval import evaluate as eval_ret
eval_ret(4)


In [ ]:
from evaluation.eval_faithfulness import main as eval_full
eval_full()


In [ ]:
# Ablation: A (LLM only) vs B (RAG) vs C (multi-agent). Writes a CSV.
from evaluation.ablation import main as run_ablation
run_ablation()


## 10. Train the binary abuse detector (optional, heavier)
Fine-tunes DistilBERT on Jigsaw. Run from a terminal or here if you have a GPU.
```bash
python train/train_abuse_clf.py
python -m evaluation.eval_classifier
```


## 11. Demo
```bash
streamlit run app/streamlit_app.py
```

---
**Reminder:** prototype only — not a replacement for professional or emergency services.
